# 03 - 常用网络层

## 学习目标

1. 掌握 `Conv2d` 的参数和输出尺寸计算公式
2. 理解 `ConvTranspose2d` 的上采样作用
3. 掌握各种池化层：`MaxPool2d`、`AvgPool2d`、`AdaptiveAvgPool2d`
4. 理解归一化层：`BatchNorm2d`、`LayerNorm`、`GroupNorm`、`InstanceNorm2d`
5. 掌握 `Dropout` 在训练和评估模式下的行为差异
6. 熟悉常用激活函数

---

In [ ]:
import torch
import torch.nn as nn

## 1. Conv2d - 二维卷积

### 关键参数

```python
nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, groups=1, bias=True)
```

### 输出尺寸公式

$$H_{out} = \lfloor \frac{H_{in} + 2 \times padding - dilation \times (kernel\_size - 1) - 1}{stride} + 1 \rfloor$$

**简化记忆**（无 dilation 时）：
$$H_{out} = \lfloor \frac{H_{in} + 2p - k}{s} + 1 \rfloor$$

In [ ]:
# Conv2d 基本使用
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)

x = torch.randn(1, 3, 32, 32)  # [batch, channels, height, width]
out = conv(x)

print(f"输入形状: {x.shape}")
print(f"输出形状: {out.shape}")
print(f"\n卷积核参数:")
print(f"  weight: {conv.weight.shape}  (out_ch, in_ch, kH, kW)")
print(f"  bias:   {conv.bias.shape}    (out_ch,)")
print(f"\n公式验证: (32 + 2*1 - 3) / 1 + 1 = 32")

In [ ]:
# 不同 stride 和 padding 的效果
configs = [
    {'kernel_size': 3, 'stride': 1, 'padding': 0},
    {'kernel_size': 3, 'stride': 1, 'padding': 1},  # same padding
    {'kernel_size': 3, 'stride': 2, 'padding': 1},  # 下采样
    {'kernel_size': 5, 'stride': 1, 'padding': 2},  # same padding (k=5)
]

x = torch.randn(1, 3, 32, 32)
print(f"输入: {x.shape}")
print()
for cfg in configs:
    conv = nn.Conv2d(3, 16, **cfg)
    out = conv(x)
    print(f"k={cfg['kernel_size']}, s={cfg['stride']}, p={cfg['padding']}  ->  输出: {out.shape}")

**常用技巧：**

- `padding = kernel_size // 2`（stride=1 时保持尺寸不变，即 "same" padding）
- `stride=2` 可以替代池化层进行下采样

## 2. ConvTranspose2d - 转置卷积（上采样）

转置卷积用于**上采样**，常见于生成器（GAN）和分割网络（U-Net）。

输出尺寸公式：
$$H_{out} = (H_{in} - 1) \times stride - 2 \times padding + dilation \times (kernel\_size - 1) + output\_padding + 1$$

In [ ]:
# ConvTranspose2d: 上采样
deconv = nn.ConvTranspose2d(16, 8, kernel_size=3, stride=2, padding=1, output_padding=1)

x = torch.randn(1, 16, 16, 16)
out = deconv(x)

print(f"输入形状: {x.shape}")
print(f"输出形状: {out.shape}")
print(f"\n上采样 2 倍: 16x16 -> 32x32")
print(f"公式: (16-1)*2 - 2*1 + 1*(3-1) + 1 + 1 = 32")

## 3. 池化层

池化层用于**下采样**，减少空间维度、降低计算量、增强平移不变性。

In [ ]:
x = torch.randn(1, 16, 32, 32)
print(f"输入形状: {x.shape}\n")

# MaxPool2d: 最大池化
max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
out1 = max_pool(x)
print(f"MaxPool2d(2, 2):        {out1.shape}")

# AvgPool2d: 平均池化
avg_pool = nn.AvgPool2d(kernel_size=2, stride=2)
out2 = avg_pool(x)
print(f"AvgPool2d(2, 2):        {out2.shape}")

# AdaptiveAvgPool2d: 自适应平均池化（只需指定输出大小）
adaptive_pool = nn.AdaptiveAvgPool2d(output_size=(1, 1))
out3 = adaptive_pool(x)
print(f"AdaptiveAvgPool2d(1,1): {out3.shape}")

# 常见用法: 全局平均池化后 squeeze
out3_flat = out3.view(out3.size(0), -1)
print(f"squeeze 后:              {out3_flat.shape}")

**关键观察：**

- `MaxPool2d` / `AvgPool2d`：需要手动计算输出尺寸
- `AdaptiveAvgPool2d(1, 1)`：不管输入多大，输出都是 1x1（全局平均池化，非常常用）

## 4. Linear - 全连接层

In [ ]:
# Linear 层
linear = nn.Linear(in_features=256, out_features=10, bias=True)

x = torch.randn(4, 256)
out = linear(x)

print(f"输入形状:  {x.shape}")
print(f"输出形状:  {out.shape}")
print(f"weight:   {linear.weight.shape}  (out_features, in_features)")
print(f"bias:     {linear.bias.shape}")
print(f"\n计算: y = x @ W^T + b")

## 5. 归一化层

归一化层通过标准化中间特征来**加速训练**、**稳定梯度**。

| 归一化方式 | 归一化维度 | 典型场景 |
|-----------|-----------|----------|
| BatchNorm | 对 batch 维度归一化 | CNN |
| LayerNorm | 对特征维度归一化 | Transformer / RNN |
| InstanceNorm | 对单个样本的空间维度归一化 | 风格迁移 |
| GroupNorm | 对通道分组归一化 | 小 batch 场景 |

### BatchNorm2d

In [ ]:
# BatchNorm2d: 对每个通道在 batch 维度上归一化
bn = nn.BatchNorm2d(num_features=16)

x = torch.randn(8, 16, 32, 32)  # [batch, channels, H, W]

# 训练模式: 使用当前 batch 的均值和方差
bn.train()
out_train = bn(x)
print("训练模式:")
print(f"  输出均值 ≈ 0: {out_train.mean(dim=(0, 2, 3))[:3].tolist()}")
print(f"  running_mean: {bn.running_mean[:3].tolist()}")
print(f"  running_var:  {bn.running_var[:3].tolist()}")

# 评估模式: 使用 running_mean 和 running_var
bn.eval()
out_eval = bn(x)
print("\n评估模式:")
print(f"  使用 running_mean 和 running_var（不更新）")
print(f"  输出与训练模式不同: {not torch.allclose(out_train, out_eval)}")

**BatchNorm 的 train vs eval：**

- **训练模式**：使用当前 batch 的均值/方差，并更新 `running_mean` / `running_var`
- **评估模式**：使用训练时积累的 `running_mean` / `running_var`，不再更新

### 四种归一化对比

In [ ]:
# 对比四种归一化层
x = torch.randn(4, 16, 8, 8)  # [batch, channels, H, W]

norms = {
    'BatchNorm2d':    nn.BatchNorm2d(16),
    'LayerNorm':      nn.LayerNorm([16, 8, 8]),
    'InstanceNorm2d': nn.InstanceNorm2d(16),
    'GroupNorm':      nn.GroupNorm(num_groups=4, num_channels=16),
}

print(f"输入形状: {x.shape}\n")
for name, norm in norms.items():
    out = norm(x)
    print(f"{name:18s}  输出形状: {out.shape}")

## 6. Dropout

Dropout 在训练时**随机置零**一部分神经元，是常用的正则化手段。

**关键点：** 训练时 dropout 生效，评估时 dropout 关闭（输出不变）。

In [ ]:
dropout = nn.Dropout(p=0.5)
x = torch.ones(1, 10)

# 训练模式: 随机置零，且放大未被置零的值
dropout.train()
out_train = dropout(x)
print("训练模式:")
print(f"  输入: {x}")
print(f"  输出: {out_train}")
print(f"  注意: 未被置零的值被放大到 {1.0 / (1 - 0.5):.1f} 倍")

# 评估模式: 不做任何操作
dropout.eval()
out_eval = dropout(x)
print("\n评估模式:")
print(f"  输入: {x}")
print(f"  输出: {out_eval}")
print(f"  输入输出完全一致: {torch.equal(x, out_eval)}")

**放大倍数的原因：**

训练时以概率 p 置零，为了保持期望值不变，未被置零的值乘以 $\frac{1}{1-p}$。这样训练和评估时的期望输出一致。

## 7. 激活函数

激活函数为网络引入**非线性**，没有激活函数，多层线性层等价于一个线性层。

In [ ]:
# 常用激活函数对比
x = torch.linspace(-3, 3, 7)

activations = {
    'ReLU':      nn.ReLU(),
    'LeakyReLU': nn.LeakyReLU(0.1),
    'GELU':      nn.GELU(),
    'Sigmoid':   nn.Sigmoid(),
    'Tanh':      nn.Tanh(),
}

print(f"输入: {x.tolist()}\n")
for name, act in activations.items():
    out = act(x)
    print(f"{name:12s}: {[round(v, 3) for v in out.tolist()]}")

In [ ]:
# Softmax: 将向量转换为概率分布
softmax = nn.Softmax(dim=-1)
logits = torch.tensor([[2.0, 1.0, 0.1]])
probs = softmax(logits)

print(f"logits: {logits}")
print(f"probs:  {probs}")
print(f"sum:    {probs.sum().item():.4f}  (概率之和为 1)")

**激活函数选择建议：**

| 激活函数 | 特点 | 适用场景 |
|---------|------|----------|
| ReLU | 简单高效，可能有 dead neuron 问题 | CNN 默认选择 |
| LeakyReLU | 解决 dead neuron | 当 ReLU 效果不好时 |
| GELU | 平滑版 ReLU | Transformer (BERT, GPT) |
| Sigmoid | 输出 (0, 1) | 二分类输出层 |
| Tanh | 输出 (-1, 1) | RNN 隐藏层 |
| Softmax | 输出概率分布 | 多分类输出层 |

## 8. 练习题

### 练习：搭建一个完整的 CNN

用本节学到的层搭建一个 CNN，处理 MNIST（1x28x28 → 10 类）：

结构要求：
- Conv2d(1, 32, 3, padding=1) → BatchNorm2d → ReLU → MaxPool2d(2)
- Conv2d(32, 64, 3, padding=1) → BatchNorm2d → ReLU → MaxPool2d(2)
- AdaptiveAvgPool2d(1)
- Linear(64, 10)

In [ ]:
class MyCNN(nn.Module):
    """处理 MNIST 的 CNN。"""

    def __init__(self):
        super().__init__()
        # TODO: 定义各层
        pass

    def forward(self, x):
        # TODO: 实现前向传播
        pass


# 测试代码
# model = MyCNN()
# x = torch.randn(4, 1, 28, 28)
# output = model(x)
# print(f"输出形状: {output.shape}")  # 应为 [4, 10]
# print(f"参数量: {sum(p.numel() for p in model.parameters()):,}")

## 9. 小结

### 层类型速查

| 类型 | 常用层 | 关键参数 |
|------|--------|----------|
| 卷积 | `Conv2d`, `ConvTranspose2d` | in/out_channels, kernel_size, stride, padding |
| 池化 | `MaxPool2d`, `AvgPool2d`, `AdaptiveAvgPool2d` | kernel_size / output_size |
| 全连接 | `Linear` | in/out_features |
| 归一化 | `BatchNorm2d`, `LayerNorm`, `GroupNorm`, `InstanceNorm2d` | num_features / normalized_shape |
| 正则化 | `Dropout` | p |
| 激活 | `ReLU`, `LeakyReLU`, `GELU`, `Sigmoid`, `Tanh`, `Softmax` | - |

### 关键要点

1. **Conv2d 输出尺寸公式**一定要记住
2. **BatchNorm 和 Dropout** 在训练和评估模式下行为不同，必须正确切换
3. `AdaptiveAvgPool2d(1)` 是全局平均池化的标准写法
4. 激活函数选择：CNN 用 ReLU，Transformer 用 GELU

### 下一步

在下一个 notebook 中，我们将学习 `nn.Module` 提供的各种 **API 方法**（模式切换、设备转移、参数遍历等）。